# 02 — Silver Cleaning & Data Quality

Bronze JSONL → **Spark** Silver Parquet + DQ reports + **DuckDB** validation. **No Gold. No modeling.**

- Primary engine: **PySpark** (`SILVER_ENGINE = "spark"`). Spark is the official engine; pandas fallback is available only manually (`allow_fallback=True`) and must not be triggered silently.
- Event absence ≠ missing data (e.g. no pit rows → handled in Gold).
- Outliers flagged; only domain-invalid values removed.
- Run after Bronze with the **same** `USE_GOOGLE_DRIVE` setting as notebook 01.


## Colab setup (required every session)

Identical to `00` and `01`: clone, `pip install -e .`, Drive mount, then import `openf1_pipeline`.


In [2]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Start Spark


In [3]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


INFO:openf1_pipeline.utils.spark:Spark session started: openf1-big-data-pipeline


Spark version: 4.0.2


## Bronze prerequisites


In [4]:
from pathlib import Path

from openf1_pipeline.config import (
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.silver.build_silver import run_silver_cleaning
from openf1_pipeline.utils.cleanup import clean_silver_layer_outputs

SILVER_ENGINE = "spark"  # manual fallback only: "pandas" with allow_fallback=True
ALLOW_FALLBACK = False
CLEAR_SILVER_OUTPUTS = True

BRONZE_DIR = get_bronze_dir()
SILVER_DIR = get_silver_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
MANIFESTS_DIR = get_manifests_dir()

print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", BRONZE_DIR)
print("SILVER_DIR:", SILVER_DIR)
print("DATA_QUALITY_REPORTS_DIR:", DATA_QUALITY_REPORTS_DIR)

jsonl_files = list(BRONZE_DIR.rglob("*.jsonl")) if BRONZE_DIR.is_dir() else []
manifest_path = MANIFESTS_DIR / "ingestion_manifest.csv"
row_counts_path = DATA_QUALITY_REPORTS_DIR / "bronze_row_counts.csv"

print(f"Bronze JSONL files found: {len(jsonl_files)}")
print("ingestion_manifest.csv:", manifest_path.exists(), manifest_path)
print("bronze_row_counts.csv:", row_counts_path.exists(), row_counts_path)

if not jsonl_files:
    raise FileNotFoundError(
        f"Bronze data not found at {BRONZE_DIR}. "
        "Run 01_ingestion_bronze.ipynb first with the same USE_GOOGLE_DRIVE setting."
    )


OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
DATA_QUALITY_REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
Bronze JSONL files found: 16
ingestion_manifest.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/ingestion_manifest.csv
bronze_row_counts.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_row_counts.csv


## Clean Silver outputs (required before fresh Spark run)

Removes prior Spark Parquet **directories**, Silver DQ CSVs, and DuckDB Silver reports.
Does not delete Bronze data.


In [5]:
if CLEAR_SILVER_OUTPUTS:
    print("Cleaning Silver layer outputs...")
    clean_silver_layer_outputs(silver_dir=SILVER_DIR, data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR)
else:
    print("Skipping Silver cleanup (CLEAR_SILVER_OUTPUTS=False).")


INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/data/silver (10 items removed)
INFO:openf1_pipeline.utils.cleanup:Silver layer cleanup complete: {'silver_data': 10, 'silver_reports': 0}


Cleaning Silver layer outputs...


## Run Silver cleaning (Spark-first)


In [6]:
outputs = run_silver_cleaning(
    bronze_dir=BRONZE_DIR,
    silver_dir=SILVER_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
    engine=SILVER_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)
outputs["summary"]


/content/openf1-big-data-pipeline/src/openf1_pipeline/silver/build_silver_spark.py:506: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  missingness_before = pd.concat(
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved meetings (25 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved sessions (123 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved drivers (40 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved laps (2031 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved pit (62 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved weather (303 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved position (927 r

{'engine': 'spark',
 'tables_loaded': 10,
 'total_rows_before': 3690,
 'total_rows_after': 3690,
 'session_result_rows_after': 40}

## DuckDB validation (Silver Parquet)


In [7]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_silver_with_duckdb,
)

silver_duckdb = validate_silver_with_duckdb(SILVER_DIR)
duckdb_silver_paths = save_duckdb_validation_reports(
    silver_duckdb, DATA_QUALITY_REPORTS_DIR, prefix="silver"
)
duckdb_silver_paths


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv (10 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report silver_row_counts -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv (10 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv (1 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report session_result_target_support -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv (1 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_duplicate_keys.csv (0 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report session_result_duplicate_keys

{'silver_row_counts': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv',
 'session_result_target_support': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv',
 'session_result_duplicate_keys': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_duplicate_keys.csv',
 'laps_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_laps_rows_by_session_key.csv',
 'pit_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_pit_rows_by_session_key.csv',
 'weather_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_weather_rows_by_session_key.csv'}

## Inspect cleaning impact


In [8]:
import pandas as pd

impact = pd.read_csv(outputs["paths"]["silver_cleaning_impact_summary"])
inventory = pd.read_csv(outputs["paths"]["silver_table_inventory"])
rules = pd.read_csv(outputs["paths"]["silver_cleaning_rules"])

display(inventory)
display(impact[["table_name", "rows_before", "rows_after", "rows_removed", "row_removal_pct"]])
display(rules.head(20))


,table_name,row_count,column_count
0,meetings,25,19
1,sessions,123,16
2,drivers,40,13
3,laps,2031,17
4,pit,62,9
5,weather,303,11
6,position,927,6
7,race_control,139,12
8,session_result,40,12
9,starting_grid,0,7


,table_name,rows_before,rows_after,rows_removed,row_removal_pct
0,meetings,25,25,0,0.0
1,sessions,123,123,0,0.0
2,drivers,40,40,0,0.0
3,laps,2031,2031,0,0.0
4,pit,62,62,0,0.0
5,weather,303,303,0,0.0
6,position,927,927,0,0.0
7,race_control,139,139,0,0.0
8,session_result,40,40,0,0.0
9,starting_grid,0,0,0,0.0


,table_name,rule_id,rule_description,rows_before,rows_after,rows_removed,severity,rationale
0,meetings,SIL_MEETINGS,Meetings cleaning,25,25,0,medium,NaN
1,sessions,SIL_SESSIONS,Sessions cleaning,123,123,0,medium,NaN
2,drivers,SIL_DRIVERS,Drivers cleaning,40,40,0,medium,NaN
3,laps,SIL_LAPS,Laps cleaning,2031,2031,0,medium,NaN
4,pit,SIL_PIT,Pit cleaning,62,62,0,medium,NaN
5,weather,SIL_WEATHER,Weather cleaning,303,303,0,medium,NaN
6,position,SIL_POSITION,Position cleaning,927,927,0,medium,NaN
7,race_control,SIL_RACE_CONTROL,Race control cleaning,139,139,0,medium,NaN
8,session_result,SIL_SESSION_RESULT,Session result cleaning,40,40,0,medium,NaN
9,starting_grid,SIL_OPTIONAL_MISSING,starting_grid optional — no Bronze files or ze...,0,0,0,low,OpenF1 may return 404; empty schema Parquet wr...


## Missingness before vs after


In [9]:
miss_before = pd.read_csv(outputs["paths"]["silver_missingness_before"])
miss_after = pd.read_csv(outputs["paths"]["silver_missingness_after"])

display(miss_before.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))
display(miss_after.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))


,table_name,column_name,dtype,missing_count,missing_pct,non_missing_count,unique_count
72,pit,stop_duration,string,62,100.0000,0,1
98,race_control,qualifying_phase,string,139,100.0000,0,1
100,race_control,sector,bigint,126,90.6475,13,5
93,race_control,driver_number,bigint,101,72.6619,38,12
94,race_control,flag,string,84,60.4317,55,8
99,race_control,scope,string,84,60.4317,55,4
107,session_result,duration,double,18,45.0000,22,23
53,laps,i1_speed,bigint,537,26.4402,1494,85
63,laps,st_speed,bigint,172,8.4687,1859,128
108,session_result,gap_to_leader,string,2,5.0000,38,24


,table_name,column_name,dtype,missing_count,missing_pct,non_missing_count,unique_count
72,pit,stop_duration,string,62,100.0000,0,1
98,race_control,qualifying_phase,string,139,100.0000,0,1
100,race_control,sector,bigint,126,90.6475,13,5
93,race_control,driver_number,bigint,101,72.6619,38,12
94,race_control,flag,string,84,60.4317,55,8
99,race_control,scope,string,84,60.4317,55,4
107,session_result,duration,double,18,45.0000,22,23
53,laps,i1_speed,bigint,537,26.4402,1494,85
63,laps,st_speed,bigint,172,8.4687,1859,128
108,session_result,gap_to_leader,string,2,5.0000,38,24


## Duplicates & referential integrity


In [10]:
dups = pd.read_csv(outputs["paths"]["silver_duplicate_report"])
ref = pd.read_csv(outputs["paths"]["silver_referential_integrity_report"])
rejected = pd.read_csv(outputs["paths"]["silver_rejected_records_summary"])

display(dups)
display(ref)
display(rejected)


,table_name,duplicate_type,duplicate_count,duplicate_pct,stage,key_columns,duplicate_key_count,affected_rows,affected_rows_pct,status,duplicate_check_note
0,meetings,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
1,meetings,key_duplicate,NaN,NaN,before,meeting_key,0.0,0.0,0.0,checked,NaN
2,sessions,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
3,sessions,key_duplicate,NaN,NaN,before,session_key,0.0,0.0,0.0,checked,NaN
4,drivers,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
5,drivers,key_duplicate,NaN,NaN,before,"session_key,driver_number",0.0,0.0,0.0,checked,NaN
6,laps,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,Converted complex object columns to stable str...
7,laps,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0,checked,NaN
8,pit,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
9,pit,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0,checked,NaN


,child_table,parent_table,key_columns,distinct_child_keys,unmatched_rows,unmatched_pct,status,stage
0,drivers,sessions,session_key,2,0,0.0,checked,before
1,laps,sessions,session_key,2,0,0.0,checked,before
2,pit,sessions,session_key,2,0,0.0,checked,before
3,weather,sessions,session_key,2,0,0.0,checked,before
4,position,sessions,session_key,2,0,0.0,checked,before
5,race_control,sessions,session_key,2,0,0.0,checked,before
6,session_result,sessions,session_key,2,0,0.0,checked,before
7,starting_grid,sessions,session_key,0,0,0.0,skipped_empty_child,before
8,laps,drivers,"session_key,driver_number",40,0,0.0,checked,before
9,pit,drivers,"session_key,driver_number",38,0,0.0,checked,before


,table_name,rule_id,reason,rejected_count
